In [ ]:
# Two-LLM simulation (Roleplayer + Agent) for weighted relaxation
#  Use finetuned Roleplayer with BOTH SFT + DPO adapters.


import os, json, re, subprocess, sys
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from setproctitle import setproctitle
setproctitle("python")

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

# ----------------------------
# Paths / config
# ----------------------------
BENCH_PATH_CANDS = ["./bench_test_20.json", "/mnt/data/bench_test_20.json", "/mnt/data/queries_10.json"]
DATA_PATH_CANDS  = ["./data.csv", "/mnt/data/data.csv"]

AGENT_OUT_PATH = "./qwen/train.json"

MODEL_NAME = "Qwen/Qwen3-8B"
SEED = 11
rng = np.random.default_rng(SEED)

# NEW: Roleplayer LoRA adapter candidates (SFT + DPO)
ROLEPLAYER_SFT_ADAPTER_CANDS = ["./qwen3_roleplayer_sft_lora", "/mnt/data/qwen3_roleplayer_sft_lora"]
ROLEPLAYER_DPO_ADAPTER_CANDS = ["./qwen3_roleplayer_dpo_lora", "/mnt/data/qwen3_roleplayer_dpo_lora"]

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

NUMERIC_COLS = {"Year", "city mpg", "highway MPG", "MSRP"}
CATEGORICAL_COLS = {"Engine Fuel Type", "Vehicle Size", "Model"}
ALLOWED_EXTRA_COLS = list(NUMERIC_COLS | CATEGORICAL_COLS)
ALLOWED_OPS = {"==", ">=", "<="}

MAX_DIALOG_TURNS_PER_SLOT = 1
MAX_NEW_TOKENS_Q = 120
MAX_NEW_TOKENS_A = 200
MAX_NEW_TOKENS_PARSE = 240

# ----------------------------
# Load benchmark + data
# ----------------------------
bench_path = next((p for p in BENCH_PATH_CANDS if os.path.exists(p)), None)
if bench_path is None:
    raise FileNotFoundError(f"Could not find benchmark json. Looked in: {BENCH_PATH_CANDS}")

data_path = next((p for p in DATA_PATH_CANDS if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not find data.csv in ./ or /mnt/data/")

with open(bench_path, "r", encoding="utf-8") as f:
    bench = json.load(f)

df0 = pd.read_csv(data_path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# ----------------------------
# Canonicalization helpers
# ----------------------------
def _build_vocab_maps(df: pd.DataFrame, cols: List[str]):
    vocab_list = {}
    vocab_lower_to_orig = {}
    for c in cols:
        if c not in df.columns:
            continue
        vals = df[c].dropna().astype(str).unique().tolist()
        vocab_list[c] = vals
        m = {}
        for v in vals:
            vl = v.strip().lower()
            if vl and vl not in m:
                m[vl] = v
        vocab_lower_to_orig[c] = m
    return vocab_list, vocab_lower_to_orig

VOCAB_LIST, VOCAB_LOWER_TO_ORIG = _build_vocab_maps(df, list(CATEGORICAL_COLS))
MAKE_SET_LOWER = set(df["Make"].dropna().astype(str).str.strip().str.lower().unique().tolist()) if "Make" in df.columns else set()

def canonicalize_categorical(col: str, raw: str) -> str:
    s = (raw or "").strip()
    if s == "" or col not in VOCAB_LOWER_TO_ORIG:
        return s

    low = s.lower()
    m = VOCAB_LOWER_TO_ORIG[col]
    if low in m:
        return m[low]

    s2 = s
    low2 = low
    if col == "Model":
        toks = re.split(r"\s+", s)
        if toks:
            t0 = toks[0].strip().lower()
            if t0 in MAKE_SET_LOWER and len(toks) >= 2:
                s2 = " ".join(toks[1:]).strip()
                low2 = s2.lower()
                if low2 in m:
                    return m[low2]

    def _clean(x: str) -> str:
        x = x.lower()
        x = re.sub(r"[^a-z0-9\s\-\/]", " ", x)
        x = re.sub(r"\b(the|a|an|model|trim|edition)\b", " ", x)
        x = re.sub(r"\s+", " ", x).strip()
        return x

    c_low = _clean(low2)

    best = None
    best_len = -1
    for v in VOCAB_LIST.get(col, []):
        vl = _clean(str(v))
        if not vl:
            continue
        if vl in c_low or c_low in vl:
            if len(vl) > best_len:
                best = v
                best_len = len(vl)
    if best is not None:
        return str(best)

    return s

# ----------------------------
# Qwen load + adapters (SFT + DPO) + robust JSON extraction
# ----------------------------
def _ensure_peft():
    try:
        import peft  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft"])

def load_qwen_with_roleplayer_adapters(model_name: str = MODEL_NAME):
    # Prefer tokenizer from DPO > SFT > base (safe fallback)
    sft_path = next((p for p in ROLEPLAYER_SFT_ADAPTER_CANDS if os.path.exists(p)), None)
    dpo_path = next((p for p in ROLEPLAYER_DPO_ADAPTER_CANDS if os.path.exists(p)), None)

    tok_src = dpo_path or sft_path or model_name
    tok = AutoTokenizer.from_pretrained(tok_src, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )

    # Attach adapters (if present)
    _ensure_peft()
    from peft import PeftModel

    loaded_any = False
    active_adapter = None

    if sft_path is not None:
        mdl = PeftModel.from_pretrained(mdl, sft_path, adapter_name="sft")
        loaded_any = True
        active_adapter = "sft"

    if dpo_path is not None:
        if not hasattr(mdl, "peft_config"):
            # shouldn't happen because SFT path might be missing, but keep safe
            mdl = PeftModel.from_pretrained(mdl, dpo_path, adapter_name="dpo")
            loaded_any = True
            active_adapter = "dpo"
        else:
            # load as a SECOND adapter
            if hasattr(mdl, "load_adapter"):
                mdl.load_adapter(dpo_path, adapter_name="dpo")
                loaded_any = True
                active_adapter = "dpo"
            else:
                # fallback: just load DPO as the only adapter (rare old PEFT)
                mdl = PeftModel.from_pretrained(mdl, dpo_path, adapter_name="dpo")
                loaded_any = True
                active_adapter = "dpo"

    # If we have both adapters and PEFT supports combining, create a merged adapter.
    if sft_path is not None and dpo_path is not None and hasattr(mdl, "add_weighted_adapter"):
        try:
            mdl.add_weighted_adapter(
                adapters=["sft", "dpo"],
                weights=[1.0, 1.0],
                adapter_name="sft_dpo",
                combination_type="linear",
            )
            active_adapter = "sft_dpo"
        except Exception as e:
            # If combining fails, just prefer dpo (it was trained starting from sft in your pipeline)
            print(f"[WARN] add_weighted_adapter failed ({e}); using adapter='{active_adapter}'")

    # Activate roleplayer adapter if any
    if loaded_any and hasattr(mdl, "set_adapter") and active_adapter is not None:
        mdl.set_adapter(active_adapter)
        print(f"Roleplayer adapters loaded. Active adapter = {active_adapter}")
    else:
        print("No roleplayer adapters found; using base model for everything.")

    mdl.eval()
    return tok, mdl

tok, mdl = load_qwen_with_roleplayer_adapters(MODEL_NAME)

def _extract_json_robust(text: str) -> dict:
    t = text.strip()
    if "```" in t:
        parts = t.split("```")
        for i in range(len(parts)-1):
            fence = parts[i].strip().lower()
            payload = parts[i+1].strip()
            if "json" in fence or fence == "":
                lines = payload.splitlines()
                if lines and lines[0].strip().lower() == "json":
                    payload = "\n".join(lines[1:]).strip()
                try:
                    return json.loads(payload)
                except Exception:
                    t = payload
                    break
    start = t.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    depth = 0
    for i in range(start, len(t)):
        ch = t[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(t[start:i+1])
    end = t.rfind("}")
    if end != -1 and end > start:
        return json.loads(t[start:end+1])
    raise ValueError("Failed to parse JSON object from model output.")

@torch.inference_mode()
def qwen_gen_text(system_prompt: str, user_prompt: str, max_new_tokens: int = 200, enable_thinking: bool = False, use_adapter: bool = True) -> str:
    """
    use_adapter=True  -> Roleplayer behavior (uses active adapter if present)
    use_adapter=False -> Agent/extractor behavior (runs on base weights via disable_adapter when available)
    """
    messages = [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}]
    prompt_text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    )
    inputs = tok(prompt_text, return_tensors="pt", padding=True)
    inputs = {k:v.to(mdl.device) for k,v in inputs.items()}
    if "attention_mask" not in inputs or inputs["attention_mask"] is None:
        inputs["attention_mask"] = (inputs["input_ids"] != tok.pad_token_id).long()

    # If PEFT supports disabling adapters, do it for agent mode.
    if (not use_adapter) and hasattr(mdl, "disable_adapter"):
        ctx = mdl.disable_adapter()
    else:
        ctx = None

    if ctx is None:
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id
        )
    else:
        with ctx:
            out = mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id
            )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

@torch.inference_mode()
def qwen_gen_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 260, retries: int = 3, enable_thinking: bool = False, use_adapter: bool = True) -> dict:
    strict_suffixes = [
        "",
        "\n\nIMPORTANT: Output MUST start with '{' and end with '}' and contain NOTHING else.",
        "\n\nFINAL WARNING: ONLY output JSON. No markdown. No code fences. No commentary."
    ]
    last_text, last_err = None, None
    for attempt in range(retries):
        sys_prompt = system_prompt + strict_suffixes[min(attempt, len(strict_suffixes)-1)]
        txt = qwen_gen_text(sys_prompt, user_prompt, max_new_tokens=max_new_tokens, enable_thinking=enable_thinking, use_adapter=use_adapter)
        last_text = txt
        try:
            return _extract_json_robust(txt)
        except Exception as e:
            last_err = e
            continue
    raise ValueError(f"Could not parse JSON after {retries} tries. Last error: {last_err}\n\nLast model output:\n{last_text}")

# ----------------------------
# Sanitizers
# ----------------------------
def coerce_int(v) -> Optional[int]:
    if v is None:
        return None
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, float) and np.isfinite(v):
        return int(round(v))
    s = str(v).strip()
    m = re.search(r"-?\d+", s)
    if not m:
        return None
    try:
        return int(m.group(0))
    except Exception:
        return None

def sanitize_constraint(p: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not isinstance(p, dict):
        return None
    col, op, val = p.get("column"), p.get("op"), p.get("value")
    if col not in ALLOWED_EXTRA_COLS or op not in ALLOWED_OPS:
        return None
    if col in NUMERIC_COLS:
        if op in (">=", "<=", "=="):
            iv = coerce_int(val)
            if iv is None:
                return None
            return {"column": col, "op": op, "value": iv}
        return None
    if col in CATEGORICAL_COLS:
        if op != "==":
            return None
        sval = str(val).strip()
        if sval == "":
            return None
        sval = canonicalize_categorical(col, sval)
        if sval == "":
            return None
        return {"column": col, "op": "==", "value": sval}
    return None

def dedup_constraints(extras: List[Dict[str,Any]]) -> List[Dict[str,Any]]:
    seen = set()
    out = []
    for p in extras:
        k = (p["column"], p["op"], str(p["value"]))
        if k in seen:
            continue
        seen.add(k)
        out.append(p)
    return out

# ----------------------------
# "Solver" filtering
# ----------------------------
def apply_constraints(df: pd.DataFrame, base: Dict[str,str], extras: List[Dict[str,Any]]) -> pd.DataFrame:
    m = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        m &= (df[c].values == base[c])
    for p in extras:
        col, op, val = p["column"], p["op"], p["value"]
        if col not in df.columns:
            return df.iloc[0:0].copy()
        if op == "==":
            m &= (df[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (df[col].values >= int(val))
        elif op == "<=":
            m &= (df[col].values <= int(val))
        else:
            return df.iloc[0:0].copy()
    return df[m].copy()

# ----------------------------
# Weighting + recommendation
# ----------------------------
def normalize_series(x: np.ndarray) -> np.ndarray:
    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if xmax - xmin < 1e-9:
        return np.zeros_like(x, dtype=float)
    return (x - xmin) / (xmax - xmin)

def recommend_after_relaxation(
    U: pd.DataFrame,
    additional: List[Dict[str,Any]],
    weights: Dict[Tuple[str,str,str], float],
    relaxed_key: Tuple[str,str,str]
) -> Optional[Dict[str,Any]]:
    keep = [p for p in additional if (p["column"], p["op"], str(p["value"])) != relaxed_key]
    m = np.ones(len(U), dtype=bool)
    for p in keep:
        col, op, val = p["column"], p["op"], p["value"]
        if op == "==":
            m &= (U[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (U[col].values >= int(val))
        elif op == "<=":
            m &= (U[col].values <= int(val))
    cand = U[m].copy()
    if len(cand) == 0:
        return None

    scores = np.zeros(len(cand), dtype=float)
    norms = {}
    for col in ["Year","city mpg","highway MPG","MSRP"]:
        if col in U.columns:
            norms[col] = normalize_series(U[col].values)

    cand_pos = cand.index.values
    for p in additional:
        k = (p["column"], p["op"], str(p["value"]))
        w = float(weights.get(k, 0.25))
        col, op, val = p["column"], p["op"], p["value"]
        if op == "==":
            sat = (cand[col].astype(str).values == str(val)).astype(float)
            scores += w * sat
        elif op == ">=":
            if col in norms:
                scores += w * norms[col][cand_pos]
            else:
                scores += w * (cand[col].values >= int(val)).astype(float)
        elif op == "<=":
            if col in norms:
                scores += w * (1.0 - norms[col][cand_pos])
            else:
                scores += w * (cand[col].values <= int(val)).astype(float)

    msrp = cand["MSRP"].values.astype(float) if "MSRP" in cand.columns else np.zeros(len(cand))
    best_idx = np.lexsort((msrp, -scores))[0]
    row = cand.iloc[best_idx]

    def gi(k):
        return int(row[k]) if k in cand.columns and pd.notna(row[k]) else None

    return {
        "Make": row.get("Make", None),
        "Model": row.get("Model", None),
        "Year": gi("Year"),
        "Engine Fuel Type": row.get("Engine Fuel Type", None),
        "Transmission Type": row.get("Transmission Type", None),
        "Driven_Wheels": row.get("Driven_Wheels", None),
        "Vehicle Size": row.get("Vehicle Size", None),
        "Vehicle Style": row.get("Vehicle Style", None),
        "city mpg": gi("city mpg"),
        "highway MPG": gi("highway MPG"),
        "MSRP": gi("MSRP"),
    }

# ----------------------------
# Two-LLM simulation prompts
# ----------------------------
ROLEPLAYER_SYS = (
    "You are ROLEPLAYER_USER. You ONLY know the provided persona and the base query. "
    "Answer the assistant's question using ONLY information that is stated or clearly implied in the persona. "
    "IMPORTANT: When the assistant is clarifying a constraint, you should be helpful and specific: "
    "if the persona provides ANY relevant value (e.g., a specific model name, a year minimum, a budget cap), "
    "you MUST state that value explicitly in your answer. "
    "Do NOT say 'no preference' or omit the value if the persona mentions it. "
    "Only say you have no preference if the persona truly contains no information about that attribute. "
    "If the assistant's question is a bit vague, still provide the most relevant constraint value(s) from the persona. "
    "Do not invent details that are not in the persona."
)

AGENT_ASK_SYS = (
    "You are CLARIFYING_AGENT. Ask ONE short question to learn the user's requirement for the given slot/field. "
    "Do not propose relaxations. Do not ask multiple questions at once."
)

AGENT_EXTRACT_SYS = (
    "You are a strict JSON generator. Extract (1) the constraint value from the user's answer for the given slot, "
    "and (2) an importance weight in [0,1] where 1=very important (must-have) and 0=very flexible (easy to relax). "
    "Output ONLY JSON."
)

def agent_question(slot: str, base_query: str, history: List[Dict[str,str]]) -> str:
    hist_txt = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history[-6:]])
    user_prompt = f"""
Base query: {base_query}

Conversation so far:
{hist_txt}

Slot to clarify: {slot}

Ask ONE question to determine the user's requirement for {slot}. Keep it concise.
"""
    # Agent uses BASE model (no adapters)
    return qwen_gen_text(AGENT_ASK_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_Q, enable_thinking=False, use_adapter=False)

def roleplayer_answer(persona: str, base_query: str, question: str) -> str:
    user_prompt = f"""
Base query the user started with:
{base_query}

Persona (all you know):
{persona}

Assistant asked:
{question}

Answer as the user, using only persona information.
"""
    # Roleplayer uses adapters (SFT + DPO if present)
    return qwen_gen_text(ROLEPLAYER_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_A, enable_thinking=False, use_adapter=True)

def extract_constraint_and_weight(slot: str, user_answer: str) -> Tuple[Optional[Dict[str,Any]], float]:
    user_prompt = f"""
Slot: {slot}
User answer: {user_answer}

Return JSON:
{{
  "constraint": null OR {{"column": "{slot}", "op": "=="|">="|"<="| "==", "value": string|number}},
  "weight": number
}}

Rules:
- If user has no preference / doesn't care, output constraint:null and weight 0.0.
- If slot is numeric (Year, city mpg, highway MPG, MSRP):
  - Use >= for minimum requirements, <= for maximum budget, == only if explicitly exact.
  - value MUST be an integer.
- If slot is categorical (Engine Fuel Type, Vehicle Size, Model):
  - Use op == and value as string.
- weight in [0,1]: must-have ~0.9-1.0; strongly prefer ~0.7-0.9; nice-to-have ~0.4-0.7; flexible/compromise ~0.0-0.3.
Output ONLY JSON.
"""
    # Extraction uses BASE model (no adapters)
    obj = qwen_gen_json(AGENT_EXTRACT_SYS, user_prompt, max_new_tokens=MAX_NEW_TOKENS_PARSE, retries=3, enable_thinking=False, use_adapter=False)
    w = float(obj.get("weight", 0.5))
    w = max(0.0, min(1.0, w))
    c = obj.get("constraint", None)
    if c is None:
        return None, w
    c["column"] = slot
    sc = sanitize_constraint(c)
    if sc is None:
        return None, w
    return sc, w

def parse_base_from_sentence(df: pd.DataFrame, base_sentence: str) -> Dict[str,str]:
    s = base_sentence.lower()
    def match_vocab(col: str) -> str:
        vocab = sorted(df[col].dropna().astype(str).unique().tolist(), key=len, reverse=True)
        for v in vocab:
            if v.lower() in s:
                return v
        raise ValueError(f"Could not parse base field {col} from: {base_sentence}")
    return {
        "Make": match_vocab("Make"),
        "Vehicle Style": match_vocab("Vehicle Style"),
        "Transmission Type": match_vocab("Transmission Type"),
        "Driven_Wheels": match_vocab("Driven_Wheels"),
    }

# ----------------------------
# Run simulation for all benchmark queries
# ----------------------------
outputs = []

for i, b in enumerate(bench, 1):
    base_query = b["base_query_sentence"]
    persona = b.get("persona", "")

    slots = sorted({c["column"] for c in b.get("additional_constraints", [])})

    history = [{"role":"user", "content": base_query}]
    collected_constraints: List[Dict[str,Any]] = []
    weights: Dict[Tuple[str,str,str], float] = {}

    for slot in slots:
        q = agent_question(slot, base_query, history)
        history.append({"role":"assistant","content": q})

        a = roleplayer_answer(persona, base_query, q)
        history.append({"role":"user","content": a})

        c, w = extract_constraint_and_weight(slot, a)

        if c is not None:
            collected_constraints.append(c)
            k = (c["column"], c["op"], str(c["value"]))
            weights[k] = w
        else:
            weights[(slot, "NA", "NA")] = w

    collected_constraints = dedup_constraints(collected_constraints)

    base = parse_base_from_sentence(df, base_query)
    full = apply_constraints(df, base, collected_constraints)

    record = {
        "id": i,
        "base_query_sentence": base_query,
        "slots_provided_to_agent": slots,
        "dialogue": history,
        "parsed_base": base,
        "parsed_additional_constraints": collected_constraints,
        "constraint_weights": [
            {"column": p["column"], "op": p["op"], "value": p["value"],
             "weight": float(weights.get((p["column"], p["op"], str(p["value"])), 0.5))}
            for p in collected_constraints
        ],
        "status": None,
        "relaxed_constraint": None,
        "recommended_car": None
    }

    if len(full) > 0:
        cand = full.reset_index(drop=True)
        wmap = {(p["column"], p["op"], str(p["value"])): float(weights.get((p["column"], p["op"], str(p["value"])), 0.5))
                for p in collected_constraints}

        scores = np.zeros(len(cand), dtype=float)
        norms = {}
        for col in ["Year","city mpg","highway MPG","MSRP"]:
            if col in cand.columns:
                norms[col] = normalize_series(cand[col].values)

        for p in collected_constraints:
            k = (p["column"], p["op"], str(p["value"]))
            w = wmap.get(k, 0.25)
            col, op, val = p["column"], p["op"], p["value"]
            if op == "==":
                scores += w
            elif op == ">=":
                if col in norms:
                    scores += w * norms[col]
            elif op == "<=":
                if col in norms:
                    scores += w * (1.0 - norms[col])

        msrp = cand["MSRP"].values.astype(float) if "MSRP" in cand.columns else np.zeros(len(cand))
        best_idx = np.lexsort((msrp, -scores))[0]
        row = cand.iloc[best_idx]

        def gi(k):
            return int(row[k]) if k in cand.columns and pd.notna(row[k]) else None

        record["status"] = "SAT_no_relaxation"
        record["recommended_car"] = {
            "Make": row.get("Make", None),
            "Model": row.get("Model", None),
            "Year": gi("Year"),
            "Engine Fuel Type": row.get("Engine Fuel Type", None),
            "Transmission Type": row.get("Transmission Type", None),
            "Driven_Wheels": row.get("Driven_Wheels", None),
            "Vehicle Size": row.get("Vehicle Size", None),
            "Vehicle Style": row.get("Vehicle Style", None),
            "city mpg": gi("city mpg"),
            "highway MPG": gi("highway MPG"),
            "MSRP": gi("MSRP"),
        }
        outputs.append(record)
        print(f"[{i}] SAT with full query. Results={len(full)}")
        continue

    if len(collected_constraints) == 0:
        record["status"] = "UNSAT_no_constraints_collected"
        outputs.append(record)
        print(f"[{i}] UNSAT and no constraints collected.")
        continue

    wmap = {(p["column"], p["op"], str(p["value"])): float(weights.get((p["column"], p["op"], str(p["value"])), 0.5))
            for p in collected_constraints}
    relaxed_key = sorted(wmap.items(), key=lambda kv: kv[1])[0][0]
    relaxed_constraint = None
    for p in collected_constraints:
        if (p["column"], p["op"], str(p["value"])) == relaxed_key:
            relaxed_constraint = {"column": p["column"], "op": p["op"], "value": p["value"]}
            break

    U = df[(df["Make"] == base["Make"]) &
           (df["Transmission Type"] == base["Transmission Type"]) &
           (df["Driven_Wheels"] == base["Driven_Wheels"]) &
           (df["Vehicle Style"] == base["Vehicle Style"])].copy().reset_index(drop=True)

    rec = recommend_after_relaxation(U, collected_constraints, wmap, relaxed_key)

    record["status"] = "SAT_after_relaxation" if rec is not None else "UNSAT_even_after_relaxation"
    record["relaxed_constraint"] = relaxed_constraint
    record["recommended_car"] = rec

    outputs.append(record)
    print(f"[{i}] UNSAT -> relaxed {relaxed_constraint} -> {record['status']}")

os.makedirs(os.path.dirname(AGENT_OUT_PATH), exist_ok=True)
with open(AGENT_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

print(f"\nWrote outputs to: {AGENT_OUT_PATH}")

# ============================================================
# Simple benchmark comparison (relaxed constraint + recommended car)
# ============================================================
def ckey(c):
    if c is None:
        return None
    return (c.get("column"), c.get("op"), str(c.get("value")))

def car_key(car):
    if not isinstance(car, dict) or not car:
        return None
    return (car.get("Make"), car.get("Model"), str(car.get("Year")), str(car.get("MSRP")))

bench_by_base = {b["base_query_sentence"]: b for b in bench}

relax_matches = 0
car_matches = 0
n = 0

for a in outputs:
    base = a["base_query_sentence"]
    if base not in bench_by_base:
        print(f"\n[SKIP] Not found in bench: {base}")
        continue

    b = bench_by_base[base]
    n += 1

    oracle_relaxed = (
        b.get("unique_repair_constraint")
        or b.get("chosen_relaxation")
        or b.get("relaxed_constraint")
    )
    agent_relaxed = a.get("relaxed_constraint")

    relax_ok = (ckey(oracle_relaxed) == ckey(agent_relaxed))
    relax_matches += int(relax_ok)

    oracle_car = b.get("recommended_car")
    agent_car = a.get("recommended_car")

    car_ok = relax_ok and (car_key(oracle_car) == car_key(agent_car))
    car_matches += int(car_ok)

    print(f"\n=== {n} ===")
    print("Base:", base)
    print("Oracle relaxed:", oracle_relaxed)
    print("Agent  relaxed:", agent_relaxed)
    print("Relax match:", relax_ok)
    print("Oracle car key:", car_key(oracle_car))
    print("Agent  car key:", car_key(agent_car))
    print("Car match:", car_ok)

print("\n=== SUMMARY ===")
print("Compared:", n)
print("Relax matches:", relax_matches, f"({relax_matches/n:.2%})" if n else "")
print("Car matches:", car_matches, f"({car_matches/n:.2%})" if n else "")
